### `etd_rk_core.ipynb`
*Created: September 26, 2026* <br/>
This notebook implements the core framework for the ETD-RK solvers. 

In [10]:
struct SemilinearODEProblem{M, F, U, P, S}
    A::M
    f::F
    u0::U
    tspan::Tuple{Float64, Float64}
    p::P
    exact_solution::S
end 

function SemilinearODEProblem(A, f, u0, tspan, p; exact_solution = nothing)
    return SemilinearODEProblem(A, f, u0, tspan, p, nothing)
end 

SemilinearODEProblem

In [7]:
abstract type ExponentialRKAlgorithm end

struct ETDRK43 <: ExponentialRKAlgorithm 
end 

#Implement later
struct ETDRK53 <: ExponentialRKAlgorithm 
end 

In [3]:
abstract type ExponentialRKCache end

mutable struct ETDRK43Cache{M, V} <: ExponentialRKCache

    #Linear term (scaled)
    M::M  #hA
    N::M  #(h/2)A 

    #Exponential matrices (or scalars if hA is a scalar)
    E::M   #ϕ₀(hA)
    G::M   #ϕ₀(hA/2)

    #Phi matrices (full step)
    P::Vector{M}   #[ϕ₁(hA), ϕ₂(hA), ϕ₃(hA)]

    #Phi matrices (half step)
    Q::Vector{M}   #[ϕ₁(hA/2), ϕ₂(hA/2), ϕ₃(hA/2)]

    #Stages 
    K::Vector{V}   #[K₁, K₂, K₃, K₄, K₅]   
    
    #Nonlinear evals 
    F::Vector{V}   #[F₁, F₂, F₃, F₄, F₅]   

    #Weights
    B::Vector{M}   #[B₁, B₂, B₃, B₄] 
end 


#Outer constructor for cache 
function ETDRK43Cache(u0::Number)

    return ETDRK43Cache(0.0,        #M
                        0.0,        #N
                        0.0,        #E
                        0.0,        #G
                        zeros(3),   #P
                        zeros(3),   #Q
                        zeros(5),   #K
                        zeros(5),   #F
                        zeros(4)    #B
                        )
end 


function ETDRK43Cache(u0::Vector{Float64})
    """
    WARNING: Do not use the same instance to initialize multiple buffers!
             If you do this then mutating M, N, E or G will update all fields that were constructed from that instance!
    """    

    vector_buffer() = zeros(length(u0))
    matrix_buffer() = zeros(length(u0), length(u0))
    
    return ETDRK43Cache(matrix_buffer(),                  #M
                        matrix_buffer(),                  #N
                        matrix_buffer(),                  #E
                        matrix_buffer(),                  #G
                        [matrix_buffer() for _ in 1:3],   #P
                        [matrix_buffer() for _ in 1:3],   #Q
                        [vector_buffer() for _ in 1:5],   #K
                        [vector_buffer() for _ in 1:5],   #F
                        [matrix_buffer() for _ in 1:4]    #B
                        )
end 


#Write one `initialize_cache` function per algorithm 
initialize_cache(u0, alg::ETDRK43) = ETDRK43Cache(u0)

#initialize_cache(u0, alg::ETDRK53) = ETDRK53_cache(u0)
#initialize_cache(u0, alg::ETDRK32) = ETDRK32_cache(u0)

initialize_cache (generic function with 1 method)

In [8]:
mutable struct SolverState{T,U}
    t::T
    u::U
    dt::T
    u_candidate::U
    error_estimate::U
    saved_t::Vector{T}       #time values at which the solution is recorded
    saved_u::Vector{U}       #solution iterates at the saved time values
    current_step::Int     
    naccept::Int             #number of accepted steps (so far)
    nreject::Int             #number of rejected steps (so far)
end

function initialize_state(t0::Float64, u0::U, dt::Float64) where {U}

    u = copy(u0)
    
    return SolverState(t0, 
                       u, 
                       dt, 
                       copy(u), 
                       zero(u),
                       Float64[t0],
                       U[copy(u)], 
                       1,
                       0,
                       0)
end

initialize_state (generic function with 1 method)

In [9]:
struct AdaptiveOptions{T}
    abstol::T               #Absolute tolerance
    reltol::T               #Relative tolerance
    safety::T               #Safety factor
    dt_min::T               #Minimum time step
    dt_max::T               #Maximum time step
    μ_min::T                #Minimum dt update factor
    μ_max::T                #Maximum dt update factor 
    μ_c::T                  #EDTRK4(3) step update parameter
    μ_e::T                  #ETDRK4(3) step update parameter
    #See Whalen et. al. (2015), page 5, for an explanation of μ_e and μ_c
end 

function AdaptiveOptions(; abstol = 1e-6, reltol = 1e-3, safety = 0.9, dt_min = 1e-10, dt_max = 1.0, μ_min = 0.4, μ_max = 2.5, μ_c = 0.85, μ_e = 1.25)
    return AdaptiveOptions(abstol, reltol, safety, dt_min, dt_max, μ_min, μ_max, μ_c, μ_e)
end

AdaptiveOptions